In [0]:
from delta.tables import DeltaTable

# ── Configuration ──────────────────────────────────────────────
CATALOG = "dbx_ws_project"
SOURCE_SCHEMA = "01_bronze"
TARGET_SCHEMA = "02_silver"

# Mapping: Bronze Table -> (Silver Table Name, Primary Key Join Condition)
TABLE_CONFIG = {
    "customers": {
        "target": "dim_customers",
        "pk": "target.customer_id = source.customer_id"
    },
    "restaurants": {
        "target": "dim_restaurants",
        "pk": "target.restaurant_id = source.restaurant_id"
    },
    "menu_items": {
        "target": "dim_menu_items",
        # Composite Key logic for menu_items
        "pk": "target.restaurant_id = source.restaurant_id AND target.item_id = source.item_id"
    }
}

# ── Processing Loop ───────────────────────────────────────────
for bronze_name, config in TABLE_CONFIG.items():
    silver_name = config["target"]
    join_cond = config["pk"]
    
    print(f"Processing {bronze_name} → {silver_name}...")

    # 1. Read from Bronze
    source_df = spark.read.table(f"{CATALOG}.{SOURCE_SCHEMA}.{bronze_name}")

    target_fqn = f"{CATALOG}.{TARGET_SCHEMA}.{silver_name}"

    # 2. Initial Load if table doesn't exist
    if not spark.catalog.tableExists(target_fqn):
        print(f"Creating new table: {silver_name}")
        source_df.write.format("delta").saveAsTable(target_fqn)
    else:
        # 3. Upsert (Merge) logic
        print(f"Performing Upsert on: {silver_name}")
        target_delta = DeltaTable.forName(spark, target_fqn)
        
        (target_delta.alias("target")
         .merge(source_df.alias("source"), join_cond)
         .whenMatchedUpdateAll()
         .whenNotMatchedInsertAll()
         .execute())

print("\n✅ Silver layer update complete!")

Processing customers → dim_customers...
Creating new table: dim_customers
Processing restaurants → dim_restaurants...
Creating new table: dim_restaurants
Processing menu_items → dim_menu_items...
Creating new table: dim_menu_items

✅ Silver layer update complete!
